# Lab 01: ReAct Agent with LangGraph + NVIDIA NIMs

**NCP-AAI Domain:** Agent Architecture & Design (15%)

## What You'll Build
A stateful ReAct (Reason + Act) agent using LangGraph that:
- Uses NVIDIA NIMs for LLM inference
- Has access to web search and calculator tools
- Persists state across turns using MemorySaver
- Demonstrates the Thought → Action → Observation loop

## Learning Objectives
- Understand LangGraph StateGraph, nodes, and edges
- Implement the ReAct loop as a cyclic graph
- Use conditional edges to route between reasoning and tool execution
- Add checkpointing for multi-turn memory

## Prerequisites
- NVIDIA API key (free at build.nvidia.com)
- Python 3.10+

In [ ]:
# Install dependencies
!pip install -q langgraph langchain-nvidia-ai-endpoints langchain-community duckduckgo-search

In [ ]:
import os
os.environ['NVIDIA_API_KEY'] = 'nvapi-xxx'  # Replace with your key

# Optional: Enable LangSmith tracing
# os.environ['LANGCHAIN_TRACING_V2'] = 'true'
# os.environ['LANGCHAIN_API_KEY'] = 'ls__xxx'
# os.environ['LANGCHAIN_PROJECT'] = 'ncp-aai-lab-01'

## Step 1: Define State and Tools

In [ ]:
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
import operator

# --- State Definition ---
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]  # append-only
    step_count: int

# --- LLM ---
llm = ChatNVIDIA(
    model='meta/llama-3.1-70b-instruct',
    temperature=0.0,  # Deterministic for tool decisions
)

# --- Tools ---
search = DuckDuckGoSearchRun()

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression. Input: Python math expression as string."""
    try:
        result = eval(expression, {'__builtins__': {}}, {})
        return str(result)
    except Exception as e:
        return f'Error: {e}'

tools = [search, calculator]
tool_map = {t.name: t for t in tools}

# Bind tools to LLM (enables function calling)
llm_with_tools = llm.bind_tools(tools)

print('Tools registered:', [t.name for t in tools])
print('LLM:', llm.model)

## Step 2: Build the ReAct Graph

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
import json

# --- Node 1: Reasoning (LLM call) ---
def reasoning_node(state: AgentState) -> AgentState:
    """LLM reasons about the current state and decides next action."""
    response = llm_with_tools.invoke(state['messages'])
    return {
        'messages': [response],
        'step_count': state['step_count'] + 1
    }

# --- Node 2: Tool Execution ---
def tool_node(state: AgentState) -> AgentState:
    """Execute tool calls from the last AI message."""
    last_message = state['messages'][-1]
    tool_messages = []

    for tool_call in last_message.tool_calls:
        tool_name = tool_call['name']
        tool_args = tool_call['args']

        print(f'  [Tool] Calling {tool_name} with args: {tool_args}')

        if tool_name in tool_map:
            result = tool_map[tool_name].invoke(tool_args)
        else:
            result = f'Error: Tool {tool_name} not found'

        tool_messages.append(ToolMessage(
            content=str(result),
            tool_call_id=tool_call['id'],
            name=tool_name
        ))

    return {'messages': tool_messages}

# --- Conditional Edge: Should we use a tool or finish? ---
def should_continue(state: AgentState) -> str:
    last_message = state['messages'][-1]

    # If LLM made tool calls -> go to tool execution
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        return 'use_tool'

    # If max steps reached -> force end
    if state['step_count'] >= 10:
        return 'end'

    # Otherwise -> done
    return 'end'

# --- Build Graph ---
graph_builder = StateGraph(AgentState)

# Add nodes
graph_builder.add_node('reasoning', reasoning_node)
graph_builder.add_node('tools', tool_node)

# Entry point
graph_builder.set_entry_point('reasoning')

# Conditional: after reasoning, either use tool or end
graph_builder.add_conditional_edges(
    'reasoning',
    should_continue,
    {'use_tool': 'tools', 'end': END}
)

# After tool execution, always go back to reasoning
graph_builder.add_edge('tools', 'reasoning')

# Compile with memory checkpointer
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

print('Graph compiled successfully!')

## Step 3: Run the Agent

In [ ]:
# Configuration: thread_id enables multi-turn memory
config = {'configurable': {'thread_id': 'lab-01-session-1'}}

def run_agent(query: str, thread_id: str = 'lab-01-session-1'):
    config = {'configurable': {'thread_id': thread_id}}
    initial_state = {
        'messages': [HumanMessage(content=query)],
        'step_count': 0
    }

    print(f'Query: {query}')
    print('-' * 50)

    result = graph.invoke(initial_state, config)

    # Print the final answer
    final_message = result['messages'][-1]
    print(f'Steps taken: {result["step_count"]}')
    print(f'Final answer: {final_message.content}')
    return result

# Test 1: Math question (will use calculator tool)
result = run_agent('What is the square root of 144 multiplied by 7?')

In [ ]:
# Test 2: Multi-step reasoning (will use search tool)
result = run_agent('What is NVIDIA TensorRT-LLM and what GPU does it require for FP8 quantization?')

In [ ]:
# Test 3: Multi-turn memory — same thread_id, follow-up question
result = run_agent('What about INT8 quantization — which GPUs support it?', thread_id='lab-01-session-1')

# Inspect full conversation history
print('\n--- Full Conversation History ---')
state_history = list(graph.get_state_history(config))
print(f'Number of checkpoints saved: {len(state_history)}')

## Step 4: Inspect the Graph Structure

In [ ]:
# Visualize the graph (requires graphviz)
try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # Fallback: print mermaid diagram
    print(graph.get_graph().draw_mermaid())

## Exam Recap

| Concept | Value |
|---------|-------|
| State type | `TypedDict` with `Annotated[list, operator.add]` for append semantics |
| Entry point | `set_entry_point('reasoning')` |
| Loop mechanism | `add_edge('tools', 'reasoning')` creates the cycle |
| Termination | `conditional_edge` returning `END` |
| Memory | `MemorySaver()` passed to `compile(checkpointer=memory)` |
| Multi-turn | Same `thread_id` in config resumes previous state |

## Challenge Exercises
1. Add a third tool: a Python code executor that uses `exec()`
2. Add a maximum token guard: check total token count before each LLM call
3. Add interrupt() before tool execution to let a human approve each tool call
4. Change thread_id and observe that conversation history resets